In [10]:
import os
#直接使用GPT2模型来计算奖励,输入是一个文本序列，输出是该序列的奖励值。gpt+线性层
import torch
from sympy import false
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        # llm最后输出的隐藏层的维度
        self.hidden_size = config.hidden_size
        # 线性层用来对llm最后输出的隐藏层给奖励
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        # 使用正态分布初始化权重
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        # 将偏置初始化为0
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        # 给出奖励
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_head = RewardHead(self.llm.config)
        '''第二种只修改头'''
        # 冻结 GPT-2 的所有参数
        # for param in self.llm.parameters():
        #     param.requires_grad = False
        # # 将 GPT-2 设置为评估模式（影响 dropout 和 batchnorm）
        # self.llm.eval()

        # '''第三种只修改最后两三层'''
        #  # 解冻最后两层 transformer 块和最后的 LayerNorm
        # for layer in self.llm.transformer.h[:-2]:
        #     for param in layer.parameters():
        #         param.requires_grad = False



    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1] # 获取最后一层的隐藏状态，而不是输出的logits

        #第一种
        # # 给出奖励
        reward = self.reward_head(last_hidden_state).squeeze(-1)
        # sigmoid用来将奖励搞到(-1,1)范围内
        # return torch.sigmoid(reward)
        return reward  # 返回 logits，不是 sigmoid

        #第二种
        # 平均池化
        # 注意：需要根据 attention_mask 计算平均，忽略 padding 部分
        # mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        # sum_embeddings = torch.sum(last_hidden_state * mask_expanded, dim=1)
        # sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        # pooled = sum_embeddings / sum_mask  # (batch, hidden)
        # reward = self.reward_head(pooled)  # (batch, 1)
        # return reward.squeeze(-1)  # (batch, )  # 返回 logits，不是 sigmoid

model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = GPT2RewardHead(model_path)
if os.path.exists('models/reward_model_大众.pt'):
    model.load_state_dict(torch.load('models/reward_model_大众.pt'))
    print('reward_model_大众.pt loaded')


reward_model_大众.pt loaded


In [11]:
from functools import partial
#加载tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer

#定义collate
def reward_collate_fn(batch):
    # batch 是一个列表，每个元素是字典，包含 'input_ids', 'attention_mask', 'score', 'score_index' 等
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    scores = [item['score'] for item in batch]
    # 原有的 score_index 不再使用，直接重新计算：
    # 每个样本的 reward token 在原始序列中的位置就是最后一个 token
    # 但经过 padding 后，我们需要找到每个样本的最后一个有效 token 的位置
    # 方法：先 padding，然后根据 attention_mask 找到每个样本的最后一个 1 的位置
    # 注意：padding 会改变序列长度，但我们可以在 padding 后再计算。

    # 使用 tokenizer 进行 padding
    padded = tokenizer.pad(
        {'input_ids': input_ids, 'attention_mask': attention_mask},
        padding=True,
        return_tensors='pt'
    )

    # 计算每个样本的最后一个有效 token 的索引
    # attention_mask 是 (batch_size, seq_len)，每行是 0/1
    # 使用 torch.argmax 可以找到最后一个 1 的位置（因为 argmax 返回第一个最大值的索引，对于 0/1 序列，最后一个 1 的位置需要更复杂的方法）
    # 简便方法：取每行非零位置的最大索引
    seq_len = padded['attention_mask'].shape[1]
    # 创建一个索引矩阵
    indices = torch.arange(seq_len).unsqueeze(0).expand(padded['attention_mask'].shape[0], -1).to(padded['attention_mask'].device)
    # 将 attention_mask 为 0 的位置索引设为 -1
    masked_indices = indices * padded['attention_mask']
    # 取每行的最大值，就是最后一个有效 token 的索引
    score_index = masked_indices.max(dim=1).values

    # 将 scores 转换为 tensor
    scores_tensor = torch.tensor(scores, dtype=torch.float32)

    # 返回一个字典，包含 input_ids, attention_mask, score, score_index
    return {
        'input_ids': padded['input_ids'],
        'attention_mask': padded['attention_mask'],
        'score': scores_tensor,
        'score_index': score_index
    }

# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(batch, REWARD_TOKEN_ID):
    # 分词
    contexts = tokenizer(batch['context'])
    batch_size = len(contexts['input_ids'])  # 批次大小

    # 初始化得分和得分位置（长度 = batch_size）
    contexts["score"] = [0.0] * batch_size
    contexts["score_index"] = [0] * batch_size

    for i in range(batch_size):
        # 在每条文本末尾添加 reward token（eos token）
        contexts['input_ids'][i].append(REWARD_TOKEN_ID)
        contexts['attention_mask'][i].append(1)

        # 评分：label 是 1（正向）或 0（负向）
        contexts['score'][i] = float(batch['label'][i])

        # 记录 reward token 的索引（最后一个位置）
        contexts['score_index'][i] = len(contexts['input_ids'][i]) - 1

    return contexts


#清理数据长度国小的
def filter_short_samples(batch, min_length=6):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)

def load_data(data_path):
    dataset = load_dataset('json', data_files={
        'train': data_path[0],
        'valid': data_path[1],
        'test': data_path[2]
    })
    # 查看结构
    print(dataset['train'][0])
    # 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

    map_kwargs = {
        'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
        'batch_size': 512,  # 每批 512 个样本
        'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
    }
    REWARD_TOKEN_ID = tokenizer.eos_token_id
    print(REWARD_TOKEN_ID)
    # 在 map 时绑定 tokenizer
    tokenize_with_REWARD_TOKEN_ID = partial(tokenize,REWARD_TOKEN_ID = REWARD_TOKEN_ID)

    # 对训练集和验证集应用 map
    tokenized_dataset_train = dataset["train"].map(tokenize_with_REWARD_TOKEN_ID, **map_kwargs)
    tokenized_dataset_val = dataset["valid"].map(tokenize_with_REWARD_TOKEN_ID, **map_kwargs)

    print(tokenized_dataset_train[0])
    print(tokenized_dataset_val[0])


    # 对训练集和验证集应用过滤函数
    filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
    filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
    print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
    print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
    print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
    print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

    # 放入torch
    import torch
    filtered_dataset_train.set_format(type='torch')
    filtered_dataset_val.set_format(type='torch')
    print(filtered_dataset_train[0])
    print(filtered_dataset_val[0])


    return filtered_dataset_train, filtered_dataset_val, dataset["test"]

data_path = ["../SFT微调大模型/大众点评数据/train.json","../SFT微调大模型/大众点评数据/val.json","../SFT微调大模型/大众点评数据/test.json"]
filtered_dataset_train, filtered_dataset_val, dataset_test = load_data(data_path=data_path)

{'idx': 19087, 'context': '点了榛子可可华夫 9元 比想象的好吃 听介绍里面是慕斯和果酱 但是冷藏过 冰冰的 感觉像冰淇淋 不是很甜 很好吃 就是太小了 两三口就没啦', 'label': 0}
50256
{'input_ids': [163, 224, 117, 12859, 228, 162, 99, 249, 36310, 20998, 107, 20998, 107, 39355, 236, 13783, 104, 860, 17739, 225, 10545, 107, 242, 46349, 111, 164, 109, 94, 21410, 25001, 121, 28938, 225, 10263, 238, 105, 20015, 233, 163, 119, 235, 34932, 234, 165, 251, 95, 42468, 162, 227, 243, 23877, 107, 161, 240, 234, 162, 252, 250, 165, 227, 109, 220, 19526, 228, 42468, 37863, 115, 164, 245, 237, 32573, 229, 10263, 228, 108, 37863, 108, 21410, 10545, 226, 253, 164, 100, 231, 161, 225, 237, 37863, 108, 162, 115, 229, 162, 115, 233, 220, 38834, 42468, 36181, 230, 18796, 250, 10263, 122, 230, 25001, 121, 28938, 225, 10263, 108, 109, 42468, 13783, 103, 22887, 237, 12859, 228, 220, 10310, 97, 49011, 20998, 96, 22887, 109, 162, 110, 94, 161, 243, 99, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [12]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
# 将 eos token 作为 pad token
tokenizer.pad_token = tokenizer.eos_token

# data_collator = DataCollatorWithPadding(tokenizer)
dataloader_params = {
    'batch_size': 16,
    'shuffle': True,
    'collate_fn': reward_collate_fn
}
train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

batch = next(iter(train_dataloader))
print(batch.keys())

print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])
print(tokenizer.decode(batch['input_ids'][1]))
print(batch['attention_mask'][1].nonzero()[-1])

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


dict_keys(['input_ids', 'attention_mask', 'score', 'score_index'])
tensor([33566,   106, 30298,   235, 29826,    96, 28839,   101, 41468,   164,
          237,   250, 40792,   171,   120,   234, 43889,    98, 43718,   117,
          163,   110,   231,   164,    95,   104,   165,   228,   233, 37345,
           94, 21410, 28938,   100, 16764, 16764,   165,   224,    96, 20046,
          230, 21410,   165,   227,   116,   161,   243,   232,   171,   120,
          223, 33232,   235, 38834, 12859,   228, 12859,   228, 30266,    98,
          163,   224,   117, 46237,   226, 10310,   233, 16764, 16764,   165,
          227,   116,   162,    95,   227,   162,   109,    97, 32573,   246,
        26193,   234,   171,   120,   234, 33566,   106, 30298,   235, 22887,
          109, 41468, 12859,   228, 32573,   247, 20046,   230, 46479,   102,
          171,   120,   234, 32573,   246, 28839,   101,   163,   255,   231,
        10310,   119,   164,   237,   250, 50256, 50256, 50256, 50256, 5025

In [13]:
outputs = model(batch['input_ids'], batch['attention_mask'])
print(outputs.shape)
print(outputs)
print(batch['score_index'])
print(batch['score'])

torch.Size([16, 150])
tensor([[ -0.3607,  -0.3368,   0.1465,  ...,   6.8745,   6.8744,   6.9031],
        [ -0.5036,  -0.4354,   1.0023,  ..., -13.1378, -12.9744, -12.9862],
        [ -0.4724,   0.3397,   0.1061,  ...,  -1.3065,   0.3812,  -1.2180],
        ...,
        [ -0.3648,  -1.1727,  -0.5190,  ...,   6.8722,   6.8824,   6.8505],
        [ -0.8927,   0.3260,   0.2668,  ...,   7.7677,   7.7493,   7.6874],
        [ -0.4790,  -1.1340,   0.4058,  ...,   1.5586,   1.5621,   1.5791]],
       grad_fn=<SqueezeBackward1>)
tensor([146, 115, 149, 112, 110, 147, 107, 119, 149, 136, 116, 122, 117, 134,
        125, 121])
tensor([1., 0., 1., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 1.])


In [14]:
#论文里面是只训练了一个epoch，奖励模型的训练通常需要较少的epoch，因为它们通常是用来评估生成模型的输出，而不是直接生成文本。过多的训练可能会导致奖励模型过拟合，从而无法有效地评估生成模型的输出。

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
# 二分类交叉熵损失
criterion = nn.BCEWithLogitsLoss()
num_epochs = 2
model.to(device)

GPT2RewardHead(
  (llm): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50304, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D()
            (c_proj): Conv1D()
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D()
            (c_proj): Conv1D()
            (act): FastGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=768, out_features=50304, bias=False)
  )
  (reward_head): RewardHead(
    (

In [15]:
# 评估
def validate():
    model.eval()
    total_loss = 0
    for i, batch in enumerate(val_dataloader):
        # 将 tensor 字段转移到设备
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        score_index = batch['score_index'].to(device)
        target = batch['score'].to(device)
        model_inputs = {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }
        with torch.no_grad():
            # 对输出进行评分
            scores = model(**model_inputs)
            # 批次中每条数据的索引
            batch_indices = torch.arange(scores.shape[0])
            # 根据索引拿出评分，也就是reward token的评分
            score = scores[batch_indices, score_index]
            # 目标评分，0 或者 1 。
            target = target  #需要注意的是这里的评分都是大于0的，但是可以通过调整reward token的位置来让模型学会给负向样本打分更低，给正向样本打分更高。
            # 计算误差
            loss = criterion(score, target)
        total_loss += loss.item()
    print('validation loss:', total_loss / len(val_dataloader))

validate()



validation loss: 1.3967669358501187


In [16]:
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
         # 将 tensor 字段转移到设备
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        score_index = batch['score_index'].to(device)
        target = batch['score'].to(device)

        model_inputs = {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, score_index]
        target = target
        loss = criterion(score, target)
        #清空梯度 ⟶ 反向传播计算梯度 ⟶ 更新参数
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())

0.8504647612571716
1.2360098361968994
0.2954621911048889
1.0728294849395752
0.47378110885620117
0.8124597668647766
0.4920220971107483
0.5762238502502441
0.6628595590591431
0.564662516117096
0.5685837268829346
0.4899922013282776
0.4925680458545685
0.6208040714263916
0.7432258129119873
0.8109031915664673
0.6591639518737793
0.5712465643882751
0.5654711127281189
0.44515079259872437
0.5297033786773682
0.4237953722476959
0.566182017326355
0.8488308191299438
1.0493204593658447
0.5547807812690735
0.4349486231803894
0.33273929357528687
0.6137491464614868
0.6000843048095703
0.7484330534934998
0.9437158703804016
0.9482051134109497
0.6245434284210205
0.5339511632919312
0.6885281205177307
0.8801549673080444
0.49888020753860474
0.6166872382164001
0.6382405757904053
0.6368546485900879
0.5277724266052246
0.5208220481872559
0.713587760925293
0.4558108448982239
0.5106701254844666
0.7639596462249756
0.5851204991340637
0.6309993863105774
0.4650444984436035
0.4434445798397064
0.519426703453064
0.7582856416

In [17]:
#保存和结果
torch.save(model.state_dict(), 'models/reward_model_大众.pt')
from sklearn.metrics import confusion_matrix
model.to(device)
model.eval()
validate()
all_predictions = []
all_labels = []

for i, batch in enumerate(val_dataloader):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    score_index = batch['score_index'].to(device)
    target = batch['score'].to(device)
    model_inputs = {
        'input_ids': input_ids,
        'attention_mask':attention_mask
    }
    with torch.no_grad():
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, score_index]
        target = target
    predictions = (score  > 0.5).int()

    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(target.cpu().numpy())

confusion_matrix(all_labels, all_predictions)
#困惑矩阵:
# 第一行：真实标签为0的情况：756个预测正确，275个预测错误
 # 第二行：真实标签为1的情况：234个预测错误，772个预测正确


validation loss: 0.6170251125639136


array([[571,  36],
       [403, 215]])